# Qwen3.5-0.8B + FlyEmbedding-v3 — identity-preserving residual adapter

This keeps the original Qwen embedding and LM head unchanged.

`e_v3 = e_qwen + alpha * Fly(e_qwen)`

`alpha` starts at exactly zero, so the notebook verifies exact embedding identity, 100% top-1 agreement, and identical deterministic generation before training.


In [ ]:
#@title 1. Update repository, install dependencies, and preflight
import pathlib, subprocess, sys, importlib
REPO_DIR=pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','huggingface_hub','safetensors','ipywidgets','pandas'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)
SRC_DIR=REPO_DIR/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
for _name in list(sys.modules):
    if _name == 'tinycenn_lm' or _name.startswith('tinycenn_lm.'): del sys.modules[_name]
importlib.invalidate_caches()
for p in [REPO_DIR/'scripts'/'run_qwen35_flyembedding_v3.py', REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flyembedding_v3.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)
from tinycenn_lm.qwen35_flyembedding_v3 import FlyEmbeddingV3Config, install_fly_embedding_v3
print('✓ FlyEmbedding-v3 preflight OK')


In [ ]:
#@title 2. Configuration
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
RUN_MODE='quick' #@param ['quick','strong']
SEQ_LEN=128 #@param {type:'integer'}
FLY_NODES=256 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}
GRAPH_MIX_INIT=0.05 #@param {type:'number'}
MAX_RESIDUAL_SCALE=0.05 #@param {type:'number'}
LR_CORE=0.0002 #@param {type:'number'}
LR_GATE=0.0005 #@param {type:'number'}
OUTPUT_DIR=REPO_DIR/'results'/'flyembedding_v3_qwen35_08b'
print('Base:',BASE_MODEL)
print('Original Qwen embedding preserved: YES')
print('Original Qwen lm_head preserved: YES')
print('Residual gate starts at zero: YES')


In [ ]:
#@title 3. Run FlyEmbedding-v3 — live output
import subprocess, sys
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flyembedding_v3.py'),
     '--base-model',BASE_MODEL,'--run-mode',RUN_MODE,'--seq-len',str(SEQ_LEN),
     '--fly-nodes',str(FLY_NODES),'--graph-steps',str(GRAPH_STEPS),
     '--graph-mix-init',str(GRAPH_MIX_INIT),'--max-residual-scale',str(MAX_RESIDUAL_SCALE),
     '--lr-core',str(LR_CORE),'--lr-gate',str(LR_GATE),'--output-dir',str(OUTPUT_DIR)]
print('='*100); print(' '.join(cmd)); print('='*100)
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''): print(line,end='',flush=True)
rc=p.wait(); print('\nFinished, exit code',rc)
if rc: raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Results
import json, pandas as pd
from IPython.display import display
report=json.loads((OUTPUT_DIR/'report.json').read_text())
hist=pd.read_csv(OUTPUT_DIR/'training_history.csv')
print('Architecture:',report['architecture'])
print('Identity:',report['identity_embedding_check'])
print('Initial deterministic generation exact:',report['initial_generation_exact'])
print('Parameter stats:\n',json.dumps(report['parameter_stats'],indent=2))
print('Initial probe:\n',json.dumps(report['initial_probe'],indent=2))
print('Best probe:\n',json.dumps(report['best_probe'],indent=2))
print('Final probe:\n',json.dumps(report['final_probe'],indent=2))
print('QUALITY GATE:',report['quality_gate_passed'])
display(hist.tail(20))
for x in report['generation_samples']:
    print('\nUSER:',x['prompt'])
    print('QWEN:',x['qwen_reply'])
    print('FLY :',x['fly_reply'])
    print('exact=',x['exact_token_match'],'| prefix=',x['matching_prefix_tokens'],'| jaccard=',round(x['token_jaccard'],3),'| passed=',x['passed'])


In [ ]:
#@title 5. Reload adapter and compare interactively
import torch, ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flyembedding_v3 import FlyEmbeddingV3Config, install_fly_embedding_v3
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
tok=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True)
if tok.pad_token_id is None: tok.pad_token=tok.eos_token
qwen=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()
fly=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()
def adj(n):
    a=torch.zeros(n,n,dtype=torch.float32)
    for i in range(n):
        a[i,i]=1
        for s in (1,3,7,17):
            a[i,(i+s)%n]=1; a[i,(i-s)%n]=1
    return (a/a.sum(-1,keepdim=True).clamp_min(1)).to(device)
ckpt=torch.load(OUTPUT_DIR/'fly_embedding_v3_adapter.pt',map_location='cpu')
cfg=FlyEmbeddingV3Config(**ckpt['config'])
install_fly_embedding_v3(fly,cfg,adj(cfg.fly_nodes))
fly.fly_embedding_v3_core.load_state_dict(ckpt['fly_embedding_v3_core'],strict=True)
fly.eval()
def answer(model,prompt):
    text=tok.apply_chat_template([{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True)
    enc=tok(text,return_tensors='pt').to(device)
    with torch.no_grad():
        y=model.generate(**enc,max_new_tokens=128,do_sample=False,use_cache=True,pad_token_id=tok.eos_token_id)
    return tok.decode(y[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip()
prompt_box=widgets.Textarea(value='Explain why residual adapters can preserve a pretrained model.',description='Prompt:',layout=widgets.Layout(width='100%',height='90px'))
button=widgets.Button(description='Compare',button_style='primary')
out=widgets.Output()
def run(_):
    q=prompt_box.value.strip()
    if not q:return
    with out:
        clear_output()
        print('QWEN:\n',answer(qwen,q)); print('\n'+'-'*80+'\n'); print('FLY-v3:\n',answer(fly,q))
button.on_click(run)
display(prompt_box,button,out)
